# Letta / MemGPT Patterns

> **Treat your LLM's context window like RAM: let the agent actively page information in and out of a three-tier memory hierarchy (core, recall, and archival), the same way an operating system manages virtual memory.**

An LLM's context window is finite. Once it fills up, older information disappears silently. The MemGPT approach (Packer et al., 2023) reframes this limitation as a **virtual memory management** problem. Think of virtual memory on your computer: you have a small amount of fast RAM and a large, slower hard drive. The operating system swaps data between them so programs can use more memory than physically exists. MemGPT does the same for an agent's context.

The agent maintains three explicit memory tiers and uses **function calls** (instructions the model issues to run specific operations) to move data between them. An **inner monologue** (private reasoning the user never sees) lets it decide what to keep, archive, and retrieve. A **heartbeat mechanism** chains multiple memory operations in a single turn.

This notebook takes a **build-from-scratch** approach:
1. We implement a minimal MemGPT-style agent loop using only the OpenAI SDK (the Python library for calling OpenAI models). No frameworks, no Letta library.
2. We demonstrate all three memory tiers, inner monologue, heartbeat chaining, and memory pressure handling.
3. We then show how the **Letta SDK** provides these same patterns in a production-ready package.

**By the end of this notebook you'll:**
- Understand the three-tier memory model (core, recall, archival) and why it matters.
- Build a working MemGPT-style agent from scratch with self-editing memory.
- See how inner monologue and heartbeat control flow work together.
- Know when to use the from-scratch approach vs. the Letta platform.


## Key Concepts

- **Core memory (always in-context)**: Two editable text blocks ("persona" and "human") that are **always included** in the system prompt. The agent rewrites these blocks via `core_memory_replace` and `core_memory_append` function calls to keep them current.
- **Recall memory (conversation history)**: The full message history stored externally. When old messages scroll out of the context window, the agent can search them with `recall_memory_search` to find earlier conversation turns.
- **Archival memory (long-term storage)**: A searchable store (backed by a vector database or a list) for information that doesn't belong in core memory but should be retrievable later. The agent uses `archival_memory_insert` and `archival_memory_search` to manage it.
- **Inner monologue**: Before every user-visible response, the agent produces a private "thinking" step. It reasons about its memory state, decides what to store or retrieve, and plans its response. This is **not shown to the user**.
- **Heartbeat mechanism**: After a function call, the agent can request another processing step (a "heartbeat") to chain multiple operations before responding. This enables multi-step memory management in a single turn.
- **Memory pressure**: When the context window approaches its limit, the system signals the agent to summarize, archive, or discard lower-priority information. This prevents silent context overflow.
- **Self-editing memory**: The defining feature. The agent autonomously rewrites its own core memory blocks as conversations evolve, keeping its in-context knowledge accurate and relevant.


## Architecture

<p align="center">
  <img src="../../images/diagrams/26_letta_memgpt_patterns.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart TB
    subgraph Context["LLM Context Window (limited)"]
        SYS["System prompt\n+ Core Memory blocks"]
        RECENT["Recent messages\n(sliding window)"]
        SYS --> RECENT
    end

    subgraph CoreMem["Core Memory (always in-context)"]
        P["Persona block\n'I am a helpful assistant\nwho remembers everything'"]
        H["Human block\n'Alice is a vegetarian\nwho works at Google'"]
    end

    subgraph RecallMem["Recall Memory (external)"]
        R[("Full conversation\nhistory\n─────────\nSearchable by\nkeyword / date")]
    end

    subgraph ArchivalMem["Archival Memory (external)"]
        A[("Long-term facts\n& documents\n─────────\nVector search\nor keyword")]
    end

    subgraph AgentLoop["Agent Loop"]
        IM["Inner monologue\n(private thinking)"]
        FC["Function calls:\ncore_memory_replace\ncore_memory_append\nrecall_memory_search\narchival_memory_insert\narchival_memory_search"]
        HB["Heartbeat\n(request more steps)"]
        RESP["User-visible response"]
    end

    CoreMem -->|"injected into"| SYS
    IM --> FC
    FC -->|"read/write"| CoreMem
    FC -->|"search"| RecallMem
    FC -->|"insert/search"| ArchivalMem
    FC -->|"request_heartbeat=true"| HB
    HB -->|"another turn"| IM
    IM --> RESP

    style P fill:#059669,color:#fff
    style H fill:#059669,color:#fff
    style R fill:#4f46e5,color:#fff
    style A fill:#7c3aed,color:#fff
    style IM fill:#d97706,color:#fff
    style FC fill:#dc2626,color:#fff
```

</details>

**The agent loop per user message:**
1. The system prompt is assembled with current core memory blocks + recent messages.
2. The agent produces an **inner monologue** (private reasoning).
3. The agent optionally calls memory functions (edit core, search recall/archival).
4. If `request_heartbeat=true`, the loop repeats from step 2 without user input.
5. When ready, the agent emits a **user-visible response** via `send_message`.


In [ ]:
# Install required packages (run once)
%pip install -q openai python-dotenv numpy

We load environment variables and create the OpenAI client. You need an `OPENAI_API_KEY` in your `.env` file. The `dotenv` library reads it automatically.

In [ ]:
import os
import json
import textwrap
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

from openai import OpenAI

client = OpenAI()

print("\u2713 Environment loaded")
print("\u2713 OpenAI client ready")

## Core Implementation - MemGPT from Scratch

We'll build a complete MemGPT-style agent in three parts:

1. **Memory stores**: Core, recall, and archival memory with their operations.
2. **Memory tools**: OpenAI function-calling tools the agent uses to manage memory.
3. **Agent loop**: The main loop with inner monologue, heartbeat, and memory pressure.


In [ ]:
class CoreMemory:
    """Two editable text blocks always included in the system prompt.

    - persona: describes who the agent is
    - human: describes what the agent knows about the current user
    """

    def __init__(self, persona: str = "", human: str = "", max_chars: int = 2000):
        self.persona = persona
        self.human = human
        self.max_chars = max_chars  # per block

    def replace(self, block: str, old_text: str, new_text: str) -> str:
        """Replace a substring in a core memory block."""
        current = getattr(self, block)
        if old_text not in current:
            return f"ERROR: '{old_text}' not found in {block} block."
        updated = current.replace(old_text, new_text, 1)
        if len(updated) > self.max_chars:
            return f"ERROR: replacement would exceed {self.max_chars} char limit."
        setattr(self, block, updated)
        return f"OK: {block} block updated."

    def append(self, block: str, text: str) -> str:
        """Append text to a core memory block."""
        current = getattr(self, block)
        updated = current + text
        if len(updated) > self.max_chars:
            return f"ERROR: append would exceed {self.max_chars} char limit."
        setattr(self, block, updated)
        return f"OK: appended to {block} block."

    def format_for_prompt(self) -> str:
        return (
            f"<core_memory>\n"
            f"<persona>\n{self.persona}\n</persona>\n"
            f"<human>\n{self.human}\n</human>\n"
            f"</core_memory>"
        )

    def __repr__(self) -> str:
        return f"CoreMemory(persona={len(self.persona)} chars, human={len(self.human)} chars)"

`RecallMemory` stores the full conversation history outside the context window. When older messages scroll out of view, the agent searches recall memory by keyword to find them again.

In [ ]:
class RecallMemory:
    """Full conversation history stored externally, searchable by keyword."""

    def __init__(self):
        self.messages: list[dict] = []

    def add(self, role: str, content: str) -> None:
        self.messages.append({
            "role": role,
            "content": content,
            "timestamp": datetime.now().isoformat(),
        })

    def search(self, query: str, limit: int = 5) -> list[dict]:
        """Keyword search over conversation history."""
        query_lower = query.lower()
        scored = []
        for msg in self.messages:
            text = msg["content"].lower()
            score = sum(1 for word in query_lower.split() if word in text)
            if score > 0:
                scored.append((score, msg))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [msg for _, msg in scored[:limit]]

    def __len__(self) -> int:
        return len(self.messages)

`ArchivalMemory` is the long-term store for facts and documents. It works like a searchable filing cabinet. The agent inserts important information here and retrieves it later with keyword search.

In [ ]:
class ArchivalMemory:
    """Long-term searchable storage for facts, documents, and knowledge."""

    def __init__(self):
        self.entries: list[dict] = []

    def insert(self, content: str) -> str:
        self.entries.append({
            "content": content,
            "timestamp": datetime.now().isoformat(),
            "id": len(self.entries),
        })
        return f"OK: archived (id={len(self.entries) - 1}, total={len(self.entries)})."

    def search(self, query: str, limit: int = 5) -> list[dict]:
        """Keyword search over archival entries."""
        query_lower = query.lower()
        scored = []
        for entry in self.entries:
            text = entry["content"].lower()
            score = sum(1 for word in query_lower.split() if word in text)
            if score > 0:
                scored.append((score, entry))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [e for _, e in scored[:limit]]

    def __len__(self) -> int:
        return len(self.entries)


print("\u2713 CoreMemory, RecallMemory, ArchivalMemory defined")

Next we define the six tools the agent can call. These are OpenAI function-calling schemas that map to the MemGPT paper's memory operations. The tools cover core memory editing (`replace` and `append`), recall and archival search, archival insert, and `send_message` (the only way the agent can talk to the user). This is a large block of tool definitions. Scan the `name` and `description` fields to understand each tool's purpose.

We define the agent's tools in three groups. The first group handles core memory editing. These two tools (`core_memory_replace` and `core_memory_append`) let the agent rewrite its own persona and human blocks.

`core_memory_replace` lets the agent swap out a substring in a core memory block. It's how the agent corrects outdated facts (for example, when the user changes jobs).

In [ ]:
# -- Memory editing tools --
# These let the agent rewrite its own core memory blocks.

TOOLS_REPLACE = [
    {
        "type": "function",
        "function": {
            "name": "core_memory_replace",
            "description": (
                "Replace a substring in a core memory block. Use this to update "
                "outdated information about yourself (persona) or the user (human)."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "block": {
                        "type": "string",
                        "enum": ["persona", "human"],
                        "description": "Which core memory block to edit.",
                    },
                    "old_text": {
                        "type": "string",
                        "description": "The exact substring to replace (must exist in the block).",
                    },
                    "new_text": {
                        "type": "string",
                        "description": "The replacement text.",
                    },
                },
                "required": ["block", "old_text", "new_text"],
            },
        },
    },
]

print(f"\u2713 {len(TOOLS_REPLACE)} core_memory_replace tool defined")


`core_memory_append` adds new text to a core memory block. The agent uses this when it learns something new about the user or itself. We combine both tools into `TOOLS_EDIT`.

In [ ]:
# -- core_memory_append tool --

TOOLS_APPEND = [
    {
        "type": "function",
        "function": {
            "name": "core_memory_append",
            "description": (
                "Append new information to a core memory block. Use this to add "
                "newly learned facts about the user or update your persona."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "block": {
                        "type": "string",
                        "enum": ["persona", "human"],
                        "description": "Which core memory block to append to.",
                    },
                    "text": {
                        "type": "string",
                        "description": "The text to append.",
                    },
                },
                "required": ["block", "text"],
            },
        },
    },
]

print(f"\u2713 {len(TOOLS_EDIT)} memory editing tools defined"),
]

# Combine edit tools
TOOLS_EDIT = TOOLS_REPLACE + TOOLS_APPEND

print(f"\u2713 {len(TOOLS_EDIT)} memory editing tools combined")


The second group covers recall and archival operations. `recall_memory_search` finds past conversation turns. `archival_memory_insert` stores facts for the long term. `archival_memory_search` retrieves them later.

`recall_memory_search` finds past conversation turns that may have scrolled out of context. `archival_memory_insert` stores important facts for the long term. We group these two because both extend the agent's memory beyond the context window.

In [ ]:
# -- Search and archival tools --
# These let the agent look up past conversations and stored facts.

TOOLS_RECALL = [
    {
        "type": "function",
        "function": {
            "name": "recall_memory_search",
            "description": (
                "Search your conversation history (recall memory) for past messages. "
                "Use this when the user references something from earlier in the "
                "conversation that may have scrolled out of your current context."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query to find past messages.",
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Max results to return (default 5).",
                        "default": 5,
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "archival_memory_insert",
            "description": (
                "Store information in long-term archival memory. Use this for facts, "
                "preferences, or knowledge that are important but don't need to be in "
                "your always-visible core memory."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "content": {
                        "type": "string",
                        "description": "The information to archive.",
                    },
                },
                "required": ["content"],
            },
        },
    },
]

print(f"\u2713 {len(TOOLS_RECALL)} recall and archival-insert tools defined")


`archival_memory_search` retrieves stored knowledge by keyword. We combine all search tools into a single `TOOLS_SEARCH` list.

In [ ]:
# -- Archival search tool --

TOOLS_ARCHIVAL_SEARCH = [
    {
        "type": "function",
        "function": {
            "name": "archival_memory_search",
            "description": (
                "Search your long-term archival memory for stored information."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query to find relevant archived information.",
                    },
                    "limit": {
                        "type": "integer",
                        "description": "Max results to return (default 5).",
                        "default": 5,
                    },
                },
                "required": ["query"],
            },
        },
    },
]

print(f"\u2713 {len(TOOLS_SEARCH)} search and archival tools defined"),
]

# Combine search tools
TOOLS_SEARCH = TOOLS_RECALL + TOOLS_ARCHIVAL_SEARCH

print(f"\u2713 {len(TOOLS_SEARCH)} search and archival tools combined")


The third group has one tool: `send_message`. This is the only way the agent can talk to the user. Everything else it writes stays as private inner monologue. We combine all three groups into a single `MEMGPT_TOOLS` list.

In [ ]:
# -- Communication tool --
# This is the ONLY way the agent can talk to the user.
# Everything else the agent writes is private inner monologue.

TOOLS_COMM = [
    {
        "type": "function",
        "function": {
            "name": "send_message",
            "description": (
                "Send a visible message to the user. You MUST call this function "
                "to communicate with the user. Any text you generate that is NOT "
                "inside a send_message call is your private inner monologue."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "message": {
                        "type": "string",
                        "description": "The message to send to the user.",
                    },
                },
                "required": ["message"],
            },
        },
    },
]

# Assemble the full tool list
MEMGPT_TOOLS = TOOLS_EDIT + TOOLS_SEARCH + TOOLS_COMM

print(f"\u2713 {len(MEMGPT_TOOLS)} total memory tools assembled:")
for tool in MEMGPT_TOOLS:
    print(f"  \u2022 {tool['function']['name']}")


Now we build the `MemGPTAgent` class. This is the main agent loop. The system prompt template embeds core memory and pressure stats directly in the prompt, giving the agent awareness of its own memory state. The constructor sets up all three memory tiers and configures context limits.

In [ ]:
class MemGPTAgent:
    """A minimal MemGPT-style agent with three-tier memory and inner monologue.

    Implements the core MemGPT loop:
    1. Assemble system prompt with core memory.
    2. Send to LLM with memory tools.
    3. Process tool calls (memory operations).
    4. If heartbeat requested, loop back to step 2.
    5. Extract user-visible response from send_message calls.
    """

    SYSTEM_TEMPLATE = (
        "You are an AI assistant with a MemGPT-style memory system.\n"
        "\n"
        "## Your Memory System\n"
        "You have three types of memory:\n"
        "1. **Core memory** (below) -- always visible to you. Edit it to keep it current.\n"
        "2. **Recall memory** -- your full conversation history. Search it for past context.\n"
        "3. **Archival memory** -- long-term storage. Insert important facts; search when needed.\n"
        "\n"
        "## How to Communicate\n"
        "- Your raw text output is your PRIVATE inner monologue (the user cannot see it).\n"
        "- To send a message the user can see, you MUST call the `send_message` function.\n"
        "- Think before acting: use your inner monologue to reason about what you know,\n"
        "  what you need to remember, and what memory operations to perform.\n"
        "\n"
        "## Memory Management Rules\n"
        "- When the user shares personal information, update the `human` block in core memory.\n"
        "- When facts change (e.g., user moves cities), use `core_memory_replace` to update.\n"
        "- For detailed information that doesn't fit in core memory, use `archival_memory_insert`.\n"
        "- If the user asks about something from earlier, search recall or archival memory.\n"
        "- Keep core memory concise -- it's always in your context window.\n"
        "\n"
        "## Current Core Memory\n"
        "{core_memory}\n"
        "\n"
        "## Memory Stats\n"
        "- Recall memory: {recall_count} messages stored\n"
        "- Archival memory: {archival_count} entries stored\n"
        "- Context usage: {context_pct}% (memory pressure: {pressure_level})\n"
    )

The constructor sets up all three memory tiers and configures limits. `context_limit` controls how many messages stay in the sliding window. `max_heartbeats` caps how many tool-call rounds the agent can do per turn.

In [ ]:
def __init__(
    self,
    model: str = "gpt-4o-mini",
    persona: str = "I am a helpful assistant with persistent memory. I remember everything important about the people I talk to.",
    context_limit: int = 20,
    max_heartbeats: int = 5,
):
    self.client = OpenAI()
    self.model = model
    self.max_heartbeats = max_heartbeats
    self.context_limit = context_limit

    # Initialize the three memory tiers
    self.core = CoreMemory(persona=persona, human="(No information yet)")
    self.recall = RecallMemory()
    self.archival = ArchivalMemory()

    # Internal conversation buffer (sliding window for context)
    self._context_messages: list[dict] = []

    # Logging for educational purposes
    self.turn_logs: list[dict] = []

MemGPTAgent.__init__ = __init__

Next we add two helper methods to the agent. `_build_system_prompt` refreshes the system message with current memory content and pressure level. `_execute_tool` routes each tool call to the correct memory operation and returns a result string.

In [ ]:
def _build_system_prompt(self) -> str:
    """Assemble the system prompt with current core memory and stats."""
    context_pct = int(len(self._context_messages) / self.context_limit * 100)
    if context_pct > 80:
        pressure = "HIGH -- consider archiving information"
    elif context_pct > 50:
        pressure = "moderate"
    else:
        pressure = "low"

    return self.SYSTEM_TEMPLATE.format(
        core_memory=self.core.format_for_prompt(),
        recall_count=len(self.recall),
        archival_count=len(self.archival),
        context_pct=context_pct,
        pressure_level=pressure,
    )

def _execute_tool(self, name: str, args: dict) -> str:
    """Execute a memory tool and return the result string."""
    if name == "core_memory_replace":
        return self.core.replace(args["block"], args["old_text"], args["new_text"])
    elif name == "core_memory_append":
        return self.core.append(args["block"], args["text"])
    elif name == "recall_memory_search":
        results = self.recall.search(args["query"], args.get("limit", 5))
        if not results:
            return "No matching messages found in recall memory."
        return "\n".join(
            f"[{r['timestamp']}] {r['role']}: {r['content']}" for r in results
        )
    elif name == "archival_memory_insert":
        return self.archival.insert(args["content"])
    elif name == "archival_memory_search":
        results = self.archival.search(args["query"], args.get("limit", 5))
        if not results:
            return "No matching entries found in archival memory."
        return "\n".join(f"[{r['id']}] {r['content']}" for r in results)
    elif name == "send_message":
        return "Message sent."
    else:
        return f"ERROR: Unknown tool '{name}'."

MemGPTAgent._build_system_prompt = _build_system_prompt
MemGPTAgent._execute_tool = _execute_tool

The `chat` method is the heart of the MemGPT loop. It stores the user message in recall memory, trims the context window, and enters a while-loop. Inside the loop the agent reasons privately (inner monologue), calls memory tools, and can request a heartbeat for another round. The loop ends when `send_message` is called or heartbeats run out.

The `chat` method is the heart of the MemGPT loop. It stores the user message in recall memory, trims the context window, and enters a while-loop. Inside the loop the agent reasons privately (inner monologue), calls memory tools, and can request a heartbeat for another round. The loop ends when `send_message` is called or heartbeats run out.

The `chat` method sets up the context and delegates to the inner loop. It stores the user message in recall memory, trims old messages, and prepares the state dict that tracks responses, tool calls, and heartbeats.

In [ ]:
def chat(self, user_message: str, verbose: bool = False) -> str:
    """Process a user message through the MemGPT loop."""
    # Store in recall memory
    self.recall.add("user", user_message)
    self._context_messages.append({"role": "user", "content": user_message})

    # Trim context window if needed
    if len(self._context_messages) > self.context_limit:
        self._context_messages = self._context_messages[-self.context_limit:]

    state = {
        "user_response_parts": [],
        "inner_monologue_parts": [],
        "tool_calls_log": [],
        "heartbeat_count": 0,
    }

    messages = [{"role": "system", "content": self._build_system_prompt()}]
    messages.extend(self._context_messages)

    self._run_loop(messages, state, verbose)

    return self._finalize_turn(
        user_message, state["user_response_parts"],
        state["inner_monologue_parts"], state["tool_calls_log"],
        state["heartbeat_count"],
    )

MemGPTAgent.chat = chat


`_run_loop` is the core MemGPT loop. Each iteration sends the context to the LLM, captures inner monologue, processes tool calls, and decides whether to heartbeat (loop again) or stop. The loop ends when `send_message` is called or heartbeats run out.

In [ ]:
def _run_loop(self, messages: list, state: dict, verbose: bool) -> None:
    """Inner MemGPT loop: reason, call tools, heartbeat, repeat."""
    while state["heartbeat_count"] <= self.max_heartbeats:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            tools=MEMGPT_TOOLS,
            temperature=0.7,
        )

        choice = response.choices[0]
        assistant_msg = choice.message

        if assistant_msg.content:
            state["inner_monologue_parts"].append(assistant_msg.content)
            if verbose:
                print(f"  \U0001f4ad Inner monologue: {assistant_msg.content[:120]}...")

        if not assistant_msg.tool_calls:
            break

        messages.append(assistant_msg)

        request_heartbeat = False
        for tc in assistant_msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)

            if verbose:
                print(f"  \U0001f527 Tool call: {fn_name}({json.dumps(fn_args, ensure_ascii=False)[:100]})")

            result = self._execute_tool(fn_name, fn_args)
            state["tool_calls_log"].append({"name": fn_name, "args": fn_args, "result": result})

            if fn_name == "send_message":
                state["user_response_parts"].append(fn_args["message"])

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

            if fn_name != "send_message":
                request_heartbeat = True

        if request_heartbeat and not state["user_response_parts"]:
            state["heartbeat_count"] += 1
            if verbose:
                print(f"  \u2764\ufe0f Heartbeat #{state['heartbeat_count']}")
            continue
        else:
            break

MemGPTAgent._run_loop = _run_loop


`_finalize_turn` composes the user-visible response, stores it in recall memory, and logs the full trace for debugging. Separating this from the main loop keeps each cell focused.

In [ ]:
def _finalize_turn(
    self, user_message, user_response_parts,
    inner_monologue_parts, tool_calls_log, heartbeat_count,
) -> str:
    """Wrap up a turn: compose the response, store it, and log the trace."""
    final_response = "\n".join(user_response_parts) if user_response_parts else "(No response generated)"

    # Store assistant response in recall
    self.recall.add("assistant", final_response)
    self._context_messages.append({"role": "assistant", "content": final_response})

    # Log for inspection
    self.turn_logs.append({
        "user_message": user_message,
        "inner_monologue": inner_monologue_parts,
        "tool_calls": tool_calls_log,
        "response": final_response,
        "heartbeats": heartbeat_count,
    })

    return final_response

MemGPTAgent._finalize_turn = _finalize_turn


These two inspection methods let you see inside the agent's memory and reasoning. `inspect_memory` prints all three tiers. `inspect_last_turn` shows the inner monologue, tool calls, and heartbeat count from the most recent turn.

In [ ]:
def inspect_memory(self) -> None:
    """Print the current state of all three memory tiers."""
    print("=" * 60)
    print("MEMORY STATE")
    print("=" * 60)
    print(f"\n--- Core Memory ---")
    print(f"Persona: {self.core.persona}")
    print(f"Human:   {self.core.human}")
    print(f"\n--- Recall Memory ({len(self.recall)} messages) ---")
    for msg in self.recall.messages[-5:]:
        print(f"  [{msg['role']}] {msg['content'][:80]}...")
    if len(self.recall) > 5:
        print(f"  ... and {len(self.recall) - 5} earlier messages")
    print(f"\n--- Archival Memory ({len(self.archival)} entries) ---")
    for entry in self.archival.entries:
        print(f"  [{entry['id']}] {entry['content'][:80]}...")
    if not self.archival.entries:
        print("  (empty)")

def inspect_last_turn(self) -> None:
    """Show detailed trace of the last agent turn."""
    if not self.turn_logs:
        print("No turns yet.")
        return
    turn = self.turn_logs[-1]
    print("=" * 60)
    print("LAST TURN TRACE")
    print("=" * 60)
    print(f"\nUser: {turn['user_message']}")
    if turn["inner_monologue"]:
        print(f"\n\U0001f4ad Inner monologue:")
        for m in turn["inner_monologue"]:
            print(f"   {m[:200]}")
    print(f"\n\U0001f527 Tool calls ({len(turn['tool_calls'])}):")
    for tc in turn["tool_calls"]:
        args_str = json.dumps(tc["args"], ensure_ascii=False)
        print(f"   {tc['name']}({args_str[:100]})")
        print(f"   \u2192 {tc['result'][:100]}")
    print(f"\n\u2764\ufe0f Heartbeats: {turn['heartbeats']}")
    print(f"\nAgent: {turn['response']}")

MemGPTAgent.inspect_memory = inspect_memory
MemGPTAgent.inspect_last_turn = inspect_last_turn

print("\u2713 MemGPTAgent class defined")

## Usage Example - Self-Editing Memory in Action

Let's walk through a multi-turn conversation where the agent:
1. Learns about the user and updates its core memory.
2. Archives detailed information it can't fit in core memory.
3. Retrieves archived information when asked.
4. Handles contradictions by replacing outdated facts.


In [ ]:
# Create an agent
agent = MemGPTAgent(model="gpt-4o-mini")

# Turn 1: User introduces themselves
print("=" * 60)
print("TURN 1 \u2014 Introduction")
print("=" * 60)
response = agent.chat(
    "Hi! I'm Alice. I'm a data scientist at Google, and I live in San Francisco. "
    "I'm a vegetarian and my favorite cuisine is Italian.",
    verbose=True,
)
print(f"\nAgent: {response}")

Let's check what the agent stored. The `inspect_memory` method prints all three memory tiers so you can see what the agent remembered from the introduction.

In [ ]:
# See what the agent stored in its memory
agent.inspect_memory()

In this turn, the user shares detailed technical information about an ML project. Watch whether the agent archives this in long-term memory. It's too detailed for core memory, but too important to lose.

In [ ]:
# Turn 2: User shares detailed technical info
print("=" * 60)
print("TURN 2 \u2014 Detailed information (archive candidate)")
print("=" * 60)
response = agent.chat(
    "I'm working on a project using PyTorch to build a recommendation system. "
    "The model architecture uses a two-tower approach with a user encoder and "
    "an item encoder, both using transformer blocks. We're training on 50M "
    "user-item interactions from the last 6 months. The key challenge is "
    "handling cold-start users who have fewer than 5 interactions.",
    verbose=True,
)
print(f"\nAgent: {response}")

Now the user's facts change: new job and new city. This is the critical test of self-editing memory. The agent should use `core_memory_replace` to update the outdated information rather than appending conflicting facts.

In [ ]:
# Turn 3: User's facts change \u2014 agent should update core memory
print("=" * 60)
print("TURN 3 \u2014 Fact change (self-editing memory)")
print("=" * 60)
response = agent.chat(
    "Big news \u2014 I just accepted a new position! I'm leaving Google "
    "and joining OpenAI next month as a research scientist. Also, "
    "I'm moving to London for the role.",
    verbose=True,
)
print(f"\nAgent: {response}")

Let's verify the agent updated its core memory. The human block should now show the new job (OpenAI) and new location (London) instead of the old values.

In [ ]:
# Check that core memory was updated with new facts
agent.inspect_memory()

print("\n\u2713 The agent updated Alice's job and location in core memory!")

The user asks about the ML project from Turn 2. If the agent archived that information, it should search archival memory and retrieve the details. Watch for the `archival_memory_search` tool call.

In [ ]:
# Turn 4: User asks about something from earlier
print("=" * 60)
print("TURN 4 \u2014 Archival memory retrieval")
print("=" * 60)
response = agent.chat(
    "What do you remember about the ML project I was working on?",
    verbose=True,
)
print(f"\nAgent: {response}")

Let's trace the agent's reasoning for the last turn. This shows the inner monologue, which tools it called, and how many heartbeats it used. This transparency is one of MemGPT's key advantages for debugging.

In [ ]:
# Inspect the detailed trace of the last turn
# This shows the inner monologue, tool calls, and heartbeats
agent.inspect_last_turn()

## Memory Pressure & Context Management

When the context window fills up, the agent receives a pressure signal. It should then proactively archive or summarize information. Let's simulate this by using a small context limit and watching the agent manage its context.


In [ ]:
# Create an agent with a small context window to trigger pressure quickly
small_agent = MemGPTAgent(model="gpt-4o-mini", context_limit=8)

# Rapid-fire messages to fill the context
messages = [
    "My favorite color is blue and I enjoy hiking on weekends.",
    "I have two cats named Luna and Mochi.",
    "I'm learning Rust and really enjoying the borrow checker.",
    "My birthday is March 15th and I love chocolate cake.",
    "I'm training for a half marathon in October.",
    "I just started reading 'Designing Data-Intensive Applications'.",
    "My partner's name is Sam and we've been together for 3 years.",
]

for i, msg in enumerate(messages, 1):
    sep = '=' * 50
    print(f"\n{sep}")
    print(f"Turn {i} (context: {len(small_agent._context_messages)}/{small_agent.context_limit})")
    print(f"{sep}")
    print(f"User: {msg}")
    response = small_agent.chat(msg, verbose=True)
    print(f"Agent: {response[:150]}...")

# Final state
print("\n" + "=" * 50)
print("FINAL MEMORY STATE")
print("=" * 50)
small_agent.inspect_memory()

## Heartbeat Chaining - Multiple Operations Per Turn

The heartbeat mechanism lets the agent perform multiple memory operations before responding. This is essential when a single user message requires both reading from and writing to memory.


In [ ]:
# New agent for heartbeat demo
hb_agent = MemGPTAgent(model="gpt-4o-mini")

# Pre-populate archival memory with some facts
hb_agent.archival.insert("Alice's favorite restaurant is Chez Panisse in Berkeley.")
hb_agent.archival.insert("Alice is allergic to shellfish but loves sushi without shrimp.")
hb_agent.archival.insert("Alice prefers window seats and flies Delta.")

print("Pre-loaded archival memory:")
for e in hb_agent.archival.entries:
    print(f"  [{e['id']}] {e['content']}")

# Ask something that requires searching archival + updating core + responding
print("\n" + "=" * 60)
print("MULTI-STEP HEARTBEAT TURN")
print("=" * 60)
response = hb_agent.chat(
    "I'm planning a dinner out. What do you know about my food preferences "
    "and restaurant choices? Also, I should mention I'm no longer allergic "
    "to shellfish \u2014 turns out it was a misdiagnosis!",
    verbose=True,
)
print(f"\nAgent: {response}")

# Show the heartbeat chain
print(f"\nHeartbeats used: {hb_agent.turn_logs[-1]['heartbeats']}")
print(f"Tool calls in this turn: {len(hb_agent.turn_logs[-1]['tool_calls'])}")
for tc in hb_agent.turn_logs[-1]["tool_calls"]:
    print(f"  \u2022 {tc['name']}")

Note: `http://localhost:8283` is the Letta server default when you run it locally via `letta server start`. It is not a public URL; swap in your own Letta deployment host for production use.


## Production Alternative - The Letta SDK

The from-scratch implementation above illustrates the MemGPT approach. For production use, the **Letta** platform provides the same patterns with added infrastructure:

- **Persistent storage**: Core, recall, and archival memory backed by databases (PostgreSQL + pgvector).
- **Multi-agent support**: Multiple agents with shared or isolated memory.
- **REST API**: Deploy agents as services with a full API.
- **Pre-built tools**: Memory management tools, web search, code execution, and more.
- **Managed hosting**: Letta Cloud for zero-infrastructure deployment.

Below is how you'd create and interact with a Letta agent. This requires running a Letta server (`letta server` or Docker).

```python
# pip install letta

from letta import create_client

# Connect to a running Letta server (or use Letta Cloud)
client = create_client()  # defaults to http://localhost:8283

# Create an agent with custom persona and human blocks
agent_state = client.create_agent(
    name="my_assistant",
    memory=ChatMemory(
        persona="I am a helpful research assistant specializing in ML papers.",
        human="User has not introduced themselves yet.",
    ),
    llm="openai/gpt-4o-mini",
    embedding="openai/text-embedding-3-small",
)

# Chat - the agent manages its own memory automatically
response = client.send_message(
    agent_id=agent_state.id,
    role="user",
    message="Hi! I'm working on a paper about attention mechanisms.",
)

# The response includes both inner monologue and user-visible messages
for msg in response.messages:
    if msg.message_type == "internal_monologue":
        print(f"[thinking] {msg.internal_monologue}")
    elif msg.message_type == "assistant_message":
        print(f"[response] {msg.assistant_message}")

# Inspect the agent's memory
memory = client.get_in_context_messages(agent_id=agent_state.id)
core = client.get_core_memory(agent_id=agent_state.id)
print(f"Persona block: {core.get_block('persona').value}")
print(f"Human block: {core.get_block('human').value}")

# Search archival memory
results = client.get_archival_memory(agent_id=agent_state.id, query="attention")
```

The Letta SDK handles all the plumbing we built manually. It takes care of system prompt assembly, tool execution, heartbeat loops, memory persistence, and context management. You can focus on agent behavior instead.


## Discussion & Tradeoffs

### Strengths

- **Explicit memory management**: Unlike opaque "memory" layers, the agent's memory operations are visible, auditable, and debuggable. You can inspect exactly what the agent chose to remember, forget, or update.
- **Graceful context overflow**: Instead of silently losing old information when the context fills up, the agent actively manages what stays in context and what gets archived. Information is never truly lost.
- **Self-editing memory**: The agent rewrites its own core memory blocks to keep them current. When facts change (user moves cities, changes jobs), the agent updates rather than accumulating stale information.
- **Inner monologue**: The private reasoning channel lets the agent "think before speaking." It plans memory operations and responses without cluttering the user's view.
- **Heartbeat chaining**: Multi-step memory operations happen smoothly in a single turn. The user doesn't need to wait or send additional messages.

### Weaknesses

- **Token overhead**: The memory management tools, system prompt, and core memory blocks consume significant context space. For short conversations, this overhead may not be worth it.
- **LLM reliability**: The agent must correctly decide *when* and *how* to use memory tools. LLMs can forget to update core memory, make incorrect replacements, or over-archive unimportant information.
- **Keyword search limitations**: Our from-scratch implementation uses keyword matching. Production systems need vector search (finding similar items by meaning using embeddings) for reliable semantic retrieval.
- **Complexity budget**: The full MemGPT loop (inner monologue + tool calls + heartbeats) adds latency and cost. Each heartbeat is an additional LLM call.
- **Cold start**: A new agent starts with minimal core memory and no archival knowledge. It takes several turns to build up useful context.

### When to Use MemGPT Patterns

| Scenario | Recommendation |
|----------|---------------|
| Long-running personal assistant (100s of sessions) | **Excellent**: the core use case |
| Agent needs to maintain accurate, evolving user profile | **Excellent**: self-editing core memory shines here |
| Short chatbot conversations | **Overkill**: sliding window or summary memory suffices |
| Agent managing complex, hierarchical knowledge | **Good**: use archival memory for depth |
| Cost-sensitive, high-throughput application | **Consider alternatives**: heartbeats and inner monologue add API calls |
| Debugging agent behavior | **Excellent**: inner monologue provides full reasoning trace |

### MemGPT vs. Other Memory Patterns

| Aspect | MemGPT / Letta | Mem0 | Summary Memory | RAG |
|--------|---------------|------|----------------|-----|
| Memory model | 3-tier (core + recall + archival) | Flat key-value store | Compressed summary | External documents |
| Who manages memory | The agent itself | The library (automatic) | The system (periodic) | The developer |
| Self-editing | Yes (core memory rewriting) | Yes (conflict resolution) | No | No |
| Inner monologue | Yes | No | No | No |
| Context overflow handling | Explicit (archival paging) | Not addressed | Summarization | Retrieval |
| Best for | Long-running agents | Quick personalization | Cost-efficient compression | Knowledge-heavy apps |


## Further Reading

- [Packer et al., "MemGPT: Towards LLMs as Operating Systems" (2023)](https://arxiv.org/abs/2310.08560) - The original paper introducing the virtual memory approach for LLMs
- [Letta Documentation](https://docs.letta.com/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - Official docs for the production MemGPT platform
- [Letta GitHub Repository](https://github.com/letta-ai/letta) - Open-source code, examples, and community
- [Letta Python SDK on PyPI](https://pypi.org/project/letta/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - Package details and installation
- [OpenAI Function Calling Guide](https://platform.openai.com/docs/guides/function-calling?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - The API mechanism used for memory tool calls
- [Anthropic - Building Effective Agents (2025)](https://www.anthropic.com/engineering/building-effective-agents?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - Agent architecture patterns that complement memory management
- [Mem0 - Automatic Memory Layer](https://github.com/mem0ai/mem0) - Alternative: automated memory extraction without agent self-management

---

*← Previous: [25 - Mem0 Integration Patterns](../25_mem0_patterns/) · Next: [27 - Zep Memory](../27_zep_memory/) →*


## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Third core memory block
Add a 'project' block to `CoreMemory` alongside 'persona' and 'human'. Update `MemGPTAgent` to include the new block in the system prompt. Use `core_memory_replace` to write project details during a conversation and verify they persist across turns.

### Challenge 2: Heartbeat chain analysis
Run `chat()` on a complex query that requires multiple memory operations (e.g., 'save this, search for something related, then answer'). Count how many heartbeat iterations `_run_loop()` takes before calling `send_message`. Test with 5 such queries and report the average chain depth.

### Challenge 3: Archival retrieval strategies
Populate `ArchivalMemory` with 20 facts. Compare keyword-based `archival_memory_search` against a cosine-similarity search using embeddings. Measure recall on 10 test queries for each strategy. This connects to the retrieval pipeline ideas in 20 Memory Retrieval Patterns.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--26-letta-memgpt-patterns--letta-memgpt-patterns)
